Log. And next steps for tomorrow

- I think I've got a good hang of finding the contributions of convolutions
- Linear is also done (although this was hodge-podge, needs better abstraction, easier code)
- ReLU is not what I thought it would be. I was clamping non-zero contribs. But thats wrong.
  - If the output activation is 0, the contrib is 0. simple.  
  - Many times, this will happen, but for convolutions it might be confusing?
  - I'll have to check how the contribution is working out (although i do believe it would already be 0 for a ReLU)
  - because the input activation was already 0, while doing the pointwise multiplication
  - still check. the basic argument of ReLU is that if the input activation is negative, the contrib propagates as 0, else it propagates as it is


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
from fastai.vision.all import *
from mtrain.utils import *
import torch.nn.functional as F

In [ ]:
# 2. Define the Model
class SimpleMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            # First Conv block: 1 channel in -> 8 filters out
            nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1),  # Output: 14x14
            nn.ReLU(),
            # Second Conv block: 8 in -> 16 filters out
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),  # Output: 7x7
            nn.ReLU(),
            # Flatten to 1D vector: 16 filters * 7 * 7 = 784
            nn.Flatten(),
            # Final Linear layer for 10 classes
            nn.Linear(16 * 7 * 7, 10),
        )

    def forward(self, x):
        return self.layers(x)


In [ ]:
path = untar_data(URLs.MNIST)

dls = ImageDataLoaders.from_folder(
    path,
    train="training",
    valid="testing",
    item_tfms=Resize(28),
    batch_tfms=Normalize(),
    img_cls=PILImageBW,
    vocab=np.array(["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]),
)

In [ ]:
def get_learner():
    model = SimpleMNIST()
    learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
    learn.remove_cb(ProgressCallback)
    learn.model = learn.model.to("cpu")
    return learn

# Train basic model

In [ ]:
learn = get_learner()
learn.fit_one_cycle(3, lr_max=1e-2)

In [ ]:
learn.show_results()

In [ ]:
torch.save(learn.model.state_dict(), "./mnist.pt")

In [ ]:
# get the tensor to analyse
inp_tens = dls.valid.one_batch()[0][0].unsqueeze(0)
torch.save(inp_tens, "./mnist-first-input-tens.pt")

# Analyse Model

Lets start by looking at the outputs of `0`

In [ ]:

class ModelSnapshot:
    def __init__(self, model):
        self.model = model
        self.activations = {}
        self.parameters = {}  # Dictionary to store weights and biases
        self.hooks = []
        
        # Capture static parameters once during init
        self._collect_parameters()
        # Register hooks for dynamic activations
        self._register_hooks()

    def _collect_parameters(self):
        """Extracts weights and biases for all layers that possess them."""
        for name, module in self.model.named_modules():
            layer_params = {}
            
            if hasattr(module, 'weight') and module.weight is not None:
                layer_params['weight'] = module.weight.detach().cpu()
            
            if hasattr(module, 'bias') and module.bias is not None:
                layer_params['bias'] = module.bias.detach().cpu()
            
            # Only add to the dictionary if the layer actually had parameters
            if layer_params:
                self.parameters[name] = layer_params

    def _register_hooks(self):
        for name, module in self.model.named_modules():
            hook = module.register_forward_hook(self._get_hook(name))
            self.hooks.append(hook)

    def _get_hook(self, name):
        def hook_fn(module, input, output):
            # Using clone() + detach() to ensure we don't interfere with the graph
            self.activations[name] = output.detach().cpu()
        return hook_fn

    def remove(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []


def get_model_internals(model, input_tensor):
    snapshot = ModelSnapshot(model)
    model.eval()

    with torch.no_grad():
        if len(input_tensor.shape) == 3:
            input_tensor = input_tensor.unsqueeze(0)
        _ = model(input_tensor)

    # Clean up hooks but keep the data
    activations = snapshot.activations
    parameters = snapshot.parameters
    snapshot.remove()
    
    return activations, parameters

In [ ]:
def get_input_patch_indices(layer, out_y, out_x):
    """
    Returns the (y_start, y_end, x_start, x_end) indices of the input
    tensor used to calculate the output pixel at (out_y, out_x).
    """
    # Extract parameters from the PyTorch submodule
    # we handle padding by actually padding the input
    # so the convolution works out fine
    # the coordinates we return assume that it is already padded and so are relative to the padded tensor
    # if you want to send the user which coordinates were actually used
    # you should subtract the padding
    # _get_actual_pad_coords is useful here
    # note that im not handling reflection padding here
    # technically, we should simply show the bigger area for our convolutions contributions
    # since padded convolutions still contribute (unless they are zero)
    # for now, lite lera since keeping track across layers can be slightly hard
    k = layer.kernel_size
    s = layer.stride

    # Calculate top-left corner
    y_start = out_y * s[0]
    x_start = out_x * s[1]

    # Calculate bottom-right corner
    y_end = y_start + k[0]
    x_end = x_start + k[1]

    # Note: Indices might be negative if they fall into the padding zone
    return (y_start, y_end, x_start, x_end)


def _get_padded(layer, tens):
    padding_mode = layer.padding_mode
    if layer.padding_mode == "zeros":
        padding_mode = "constant"
    padded = F.pad(tens, layer._reversed_padding_repeated_twice, padding_mode)
    return padded


def conv_calculate_raw_contribs_for_single_out_pixel(
    layer, acts_layer_in, b, k, out_y, out_x
):
    # we return the patch relative to the padded input itself
    padded = _get_padded(layer, acts_layer_in)

    y0, y1, x0, x1 = get_input_patch_indices(layer, out_y, out_x)
    # we do pointwise mult for this one
    cube = padded[b, :, y0:y1, x0:x1].cpu()
    kernel = layer.weight[k].cpu()
    bias = layer.bias[k].cpu()
    pointwise = cube * kernel
    bias_for_each_component = bias / pointwise.numel()

    # this is for checking
    # verified
    calculated_activation = pointwise.sum() + bias

    contribs = pointwise + bias_for_each_component
    total_sum = pointwise.abs().sum() + bias.abs()
    contribs /= total_sum
    # we need to return the patch also
    return (
        contribs,
        calculated_activation,
        (y0, y1, x0, x1),
    )


def verify_manual_conv_works(layer, acts_layer_in, acts_layer_out):
    # go through all output activations
    # for each, find the input patch
    # for each find the
    res = torch.zeros(acts_layer_out.shape).to(torch.float32)

    for b in range(len(acts_layer_out)):
        for k in range(len(acts_layer_out[b])):
            for i in range(len(acts_layer_out[b][k])):
                for j in range(len(acts_layer_out[b][k][i])):
                    _, calculated_activation, _ = (
                        conv_calculate_raw_contribs_for_single_out_pixel(
                            layer, acts_layer_in, b, k, i, j
                        )
                    )
                    res[b][k][i][j] = calculated_activation
    assert torch.allclose(acts_layer_out, res, atol=1e-6)
    print("All good ✅")


def conv_calculate_contribs_for_all(layer, acts_layer_in, acts_layer_out, out_contribs):
    all_contribs = _get_padded(
        layer, torch.zeros(acts_layer_in.shape).to(torch.float32)
    )

    for b in range(len(acts_layer_out)):
        for k in range(len(acts_layer_out[b])):
            for i in range(len(acts_layer_out[b][k])):
                for j in range(len(acts_layer_out[b][k][i])):
                    # these are the contribs for all the input patch
                    in_contribs, calced_val, patch_coords = (
                        conv_calculate_raw_contribs_for_single_out_pixel(
                            layer, acts_layer_in, b, k, i, j
                        )
                    )
                    y0, y1, x0, x1 = patch_coords
                    # we use the same cube we used before the setting the contribs
                    # we also multiply the contribs with the actual contrib of the output neuron
                    all_contribs[b][:, y0:y1, x0:x1] += (
                        in_contribs * out_contribs[b][k][i][j]
                    )
                    if not torch.allclose(
                        calced_val, acts_layer_out[b][k][i][j], atol=1e-6
                    ):
                        print(
                            "dtypes", calced_val.dtype, acts_layer_out[b][k][i][j].dtype
                        )
                        raise Exception(
                            f"found manually calculated convolution result to be different than actual activation.\nCalculated={calced_val.item()} Actual={acts_layer_out[b][k][i][j].item()}"
                        )

    pady, padx = layer.padding
    return all_contribs[:, :, pady:-pady, padx:-padx]

In [ ]:
def linear_calculate_contribs_for_all(
    layer, acts_layer_in, acts_layer_out, out_contribs
):
    weight, bias = layer.weight, layer.bias
    all_contribs = torch.zeros(acts_layer_in.shape).to(torch.float32)

    for b in range(len(acts_layer_out)):
        for out_act_i in range(len(acts_layer_out[b])):
            pointwise = weight[out_act_i] * acts_layer_in[b]
            this_bias = bias[out_act_i]
            bias_for_each_component = this_bias / pointwise.numel()
            contribs = pointwise + bias_for_each_component
            total_sum = pointwise.abs().sum() + this_bias.abs()
            contribs /= total_sum

            all_contribs[b] += contribs * out_contribs[b][out_act_i]

    return all_contribs


def relu_calculate_contribs(out_contribs):
    # for now, we say that ReLU does not change contribs, its just an ID function, contribs flow back as is
    return out_contribs

In [ ]:
def get_contribs_for_inp(inp_tens, learn, input_ratios):
    acts, parameters = get_model_internals(learn.model, inp_tens)
    total_contribs = {}


    total_contribs["layers.5"] = input_ratios


    total_contribs["layers.4"] = linear_calculate_contribs_for_all(
        learn.model.get_submodule("layers.5"),
        acts["layers.4"],
        acts["layers.5"],
        total_contribs["layers.5"],
    )
    # flatten
    total_contribs["layers.3"] = total_contribs["layers.4"].view([1, 16, 7, 7])

    total_contribs["layers.2"] = relu_calculate_contribs(total_contribs["layers.3"])

    total_contribs["layers.1"] = conv_calculate_contribs_for_all(
        learn.model.get_submodule("layers.2"),
        acts["layers.1"],
        acts["layers.2"],
        total_contribs["layers.2"],
    )
    total_contribs["layers.0"] = relu_calculate_contribs(total_contribs["layers.1"])
    total_contribs["inputs"] = conv_calculate_contribs_for_all(
        learn.model.get_submodule("layers.0"),
        inp_tens,
        acts["layers.0"],
        total_contribs["layers.0"],
    )
    return total_contribs, acts, parameters


def zeros_with_1_at(length, idx_of_1):
    res = torch.zeros(length).to(torch.float32).unsqueeze(0).cpu()
    res[0][idx_of_1] = 1.0
    return res

In [ ]:
learn = get_learner()
learn.model.load_state_dict(torch.load("./mnist.pt", map_location="cpu"))
inp_tens = torch.load("./mnist-first-input-tens.pt", weights_only=False).to("cpu")

In [ ]:
# 9.5881 score for 9
original_output = learn.model(inp_tens)
original_output

In [ ]:
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(inp_tens, learn, final_ratios)
show_single_channel_red_green_black(
    [inp_tens[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",


In [ ]:
l0_w, l0_b = parameters['layers.0']["weight"], parameters["layers.0"]["bias"]
in_acts = inp_tens
out_acts = acts["layers.0"]
out_contribs = contribs["layers.1"]
in_contribs = contribs["inputs"]

print(acts.keys(), parameters.keys())
print(l0_w.shape, l0_b.shape, in_acts.shape, out_acts.shape, out_contribs.shape, in_contribs.shape)

In [ ]:
l0_w[1][0].sum(), l0_w[0][0]

In [ ]:
show_single_channel_red_green_black(
    [w[0].detach().numpy() for w in l0_w]
, (20,10), ncols=4, viztype="local")

In [ ]:
show_single_channel_red_green_black([in_acts[0][0].detach().numpy()], ncols=1)
# plt.imshow(in_acts[0][0].detach().numpy(), cmap="gray")

In [ ]:
show_single_channel_red_green_black(
    [a.detach().numpy() for a in out_acts[0]]
, (20,10), ncols=4, viztype="local")

In [ ]:
show_single_channel_red_green_black(
    [c.detach().numpy() for c in out_contribs[0]]
, (20,10), ncols=4)

In [ ]:
show_single_channel_red_green_black(
    [c.detach().numpy() for c in out_contribs[0]]
, (20,10), ncols=4)

## Removing the top diagonal

If we look at the contribs on the inputs, we see that the top left side diagonal of the 9 is not contributing.  
We can remove it and see if its giving the same score

In [ ]:
inp_tens = torch.load("./mnist-first-input-tens.pt", weights_only=False).to("cpu")
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(inp_tens, learn, final_ratios)
show_single_channel_red_green_black(
    [inp_tens[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)

Row 9 has very less contrib, lets remove that portion (9 row left part, which [9][:12])

In [ ]:
plt.plot(inp_tens[0][0][9])

In [ ]:
# clip the values from 4 -> 12 to the first value (which is 0 in normalized vector)
clipped = inp_tens.clone()
clipped[0][0][9][4:12] = clipped[0][0][9][0]
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(clipped, learn, final_ratios)
show_single_channel_red_green_black(
    [clipped[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)
# 9.5 -> 8.5
preds = learn.model(clipped)
print(preds)

In [ ]:
# lets actually remove the full 9th line
clipped = inp_tens.clone()
clipped[0][0][9][:] = clipped[0][0][9][0]
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(clipped, learn, final_ratios)
show_single_channel_red_green_black(
    [clipped[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)
# 8.4 from 8.5, did not change at all, the contribs are quite similar too
preds = learn.model(clipped)
print(preds)

In [ ]:
# lets remove 8 also
clipped = inp_tens.clone()
clipped[0][0][8][:] = clipped[0][0][8][0]
clipped[0][0][9][:] = clipped[0][0][9][0]
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(clipped, learn, final_ratios)
show_single_channel_red_green_black(
    [clipped[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)
# 8.4 -> 6.7 nice
preds = learn.model(clipped)
print(preds)

In [ ]:
# a majority of the contribs are coming from the downwards diagonal. Lets remove the top line also.  
# lets remove 8 also
clipped = inp_tens.clone()
clipped[0][0][7][:] = clipped[0][0][7][0]
clipped[0][0][8][:] = clipped[0][0][8][0]
clipped[0][0][9][:] = clipped[0][0][9][0]
final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(clipped, learn, final_ratios)
show_single_channel_red_green_black(
    [clipped[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)
# 8.4 -> 6.7 nice
preds = learn.model(clipped)
print(preds)

In [ ]:
clipped = inp_tens.clone()
clipped[0, 0, 13:20, 0:13] = clipped[0][0][0][0]
plt.imshow(clipped[0][0])

In [ ]:
# it is very confident on the left part going down in the circle
# lets take that out
clipped = inp_tens.clone()
clipped = inp_tens.clone()
clipped[0, 0, 13:20, 0:13] = clipped[0][0][0][0]

final_ratios = zeros_with_1_at(10, 9)
contribs, acts, parameters = get_contribs_for_inp(clipped, learn, final_ratios)
show_single_channel_red_green_black(
    [clipped[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)
# 8.4 -> 6.7 nice
preds = learn.model(clipped)
print(preds)

# Detour checking out a top loss contribs

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
losses, idxs = interp.top_losses()
# classification interpretation puts it on mps
learn.model = learn.model.to("cpu")

In [ ]:
inp_tens, category = learn.dls.valid.dataset[idxs[0]]
inp_tens.show()
inp_tens = learn.dls.test_dl([inp_tens]).one_batch()[0]
inp_tens = inp_tens.cpu()
inp_tens.shape, category

In [ ]:
contribs, acts, parameters = get_contribs_for_inp(inp_tens, learn, zeros_with_1_at(10, 7))
show_single_channel_red_green_black(
    [inp_tens[0][0].detach().numpy(), contribs["inputs"][0][0].detach().numpy()],
    (10,10),
    viztype="local",
)

In [ ]:
# ive got acts, contribs
# i can take a look at each layers inputs and outputs, along with the weights


l0_w, l0_b = parameters['layers.0']["weight"], parameters["layers.0"]["bias"]
in_acts = inp_tens
out_acts = acts["layers.0"]
out_contribs = contribs["layers.1"]
in_contribs = contribs["inputs"]

print(acts.keys(), parameters.keys())
print(l0_w.shape, l0_b.shape, in_acts.shape, out_acts.shape, out_contribs.shape, in_contribs.shape)

In [ ]:
plt.imshow(in_acts[0][0].detach().numpy(), cmap="gray")

In [ ]:
l0_w[1][0]

In [ ]:
show_single_channel_red_green_black(
    [w[0].detach().numpy() for w in l0_w]
, (20,10), ncols=4, viztype="local")

In [ ]:
show_single_channel_red_green_black(
    [a.detach().numpy() for a in out_acts[0]]
, (20,10), ncols=4, viztype="local")

In [ ]:
show_single_channel_red_green_black(
    [c.detach().numpy() for c in out_contribs[0]]
, (20,10), ncols=4)

# Scratch

How do i combine contributions within a layer?   

Lets say we have

```
A0 [8 activations] -> L0 (8x4) -> A1[4 activations] -> L1(4x2) -> A2[2 activations]
```

A single activation in `A0` is multiplied 4 times, with 4 weights in the `L0` layer.  

8 activations in A0 are multiplied to 4 slices of 8 weights each (they are all different).  (and then summed, slice wise)
This results in each activation in `A0` responsible for some contribution in the next 4 output activations.  

Now, this is true for all 8 activations in `A0`. I want the relative importance each of them have.  
The contributions are already scaled (we multiply them to the contribution each of the 4 pixels have).  

So in absolute importance, every contribution for every activation in L0 is scaled correctly.  

So `A0[0] -> 4 contributions`, `A0[1] -> 4 contributions` and so on.   

The question now, is how do i create a single number out of these 4 contributions.  


The easiest way is to add them. Does it keep the property sane. Our invariant.   


What is my goal?
- Seeing how a single value within a layer affects the final output, wrt the other values in that layer (activations of that layer)
- Now we calculate the contribution of an activation by finding its contribution to the output activation
- Now the problem is, how do you know that the relative contribution of some value which results in some output x, and the relative contribution of some value which results in output y, mean the same thing. That is, they are scaled.
  - I do this, by multiplying the relative contribution, to the output's contribution. And so, ALL values are now scaled globally. 
  - We can say globally, that this activation value, has some contribution `k` to the output. THIS is the final goal.  

So lets backtrack to the final goal
- We can say globally, that this activation value, has some contribution `k` to the output. 

I multiply by the output contribution because for multiple outputs, i want to make sure that the input contribs are scaled, among multiple input activations.  

That is, lets `0,1,2` activatations are used to give output `0` and `5,6,7` pixels are used to give output `1`. We can only compare their relative importance if we know the relative importance of the output pixels `0` and `1`.  

Thats why we multiply them.

Now the question is `0` contributes some `0.4` to output act `o0` and `0.1` to output act `o1`. These activations themselves have their relative importance, say `o0 = 0.1`, `o1 = 0.8`.  
Then `0` contributes `0.04` and `0.08` in absolute terms (to the final output) (in that layer). (note that using the decision of `abs.sum` is still contended. Im not sure if that is the right thing to do, for now, abs does make sense).  

I think, it is okay to add them for now. I dont know any other way to do it. (since they are all "scaled", we should be okay).  


There is one problem though, there is no symmetry with the last layer.   
For the last layer, I look individually at how the activations contributed for each class. I don't sum up those contributions.  
A contribution which works for 9 can be very negative for say a 4.  

Summing these contributions naively would be pretty wrong I guess.  

Instead of combining, can I just increase the number of dimensions im using to track?  

So for now, I think Im gonna add them up. Im not sure what to do other than this.  

- So finally, the total contribution of some activation is the sum of its contributions to all output activations
- Contribution of a single activation to some output activation is calculated based on the layer type